# Principal Component Analysis via SVD

**Difficulty**: 🟢 Easy

---

### Problem Statement

PCA is the first tool people reach for when looking at high-dimensional data. It is used to
compress CLIP/DINO embeddings, to visualize learned features, to whiten inputs, to analyze
activations and gradients, and to initialize low-rank adapters. It finds the orthonormal directions
along which centered data varies most. Those are the top eigenvectors of the sample covariance

$$\Sigma = \frac{1}{N-1}\,X_c^{\top} X_c, \qquad X_c = X - \bar x .$$

The numerically sound way to get them is **not** to form $\Sigma$ at all. Take the **singular value
decomposition** of the centered data instead, $X_c = U\,\mathrm{diag}(s)\,V^{\top}$.

---

### Requirements

1. **`pca_fit(X, k) -> (components, explained_variance, mean)`**
   - `X`: `(N, D)` float tensor, one sample per row. `1 <= k <= min(N, D)`.
   - `mean`: `(D,)` per-feature mean.
   - `components`: `(k, D)` principal directions as **orthonormal rows**, sorted by decreasing variance.
   - `explained_variance`: `(k,)` variance of the data along each component, using the **unbiased
     $N-1$ convention**: $\lambda_i = s_i^2/(N-1)$. These equal the top-$k$ eigenvalues of
     `torch.cov(X.T)`.
   - Compute them from the SVD of the centered data.
2. **`pca_transform(X, components, mean)`** → `(N, k)`: coordinates $Z = (X - \bar x)\,C^{\top}$.
3. **`pca_inverse_transform(Z, components, mean)`** → `(N, D)`: reconstruction $\hat X = Z\,C + \bar x$.
4. The sign of each component is arbitrary ($v$ and $-v$ are both eigenvectors). Any sign is accepted.
5. Properties your implementation should satisfy (the tests check them):
   - projected coordinates are uncorrelated, with variance $\lambda_i$ along axis $i$;
   - $\frac{1}{N-1}\sum_n \lVert x_n - \hat x_n\rVert^2 = \sum_{i>k}\lambda_i$, the variance you discarded;
   - with `k = D` the reconstruction is exact, and `pca_transform(mean)` is $0$.

---

### Constraints

- ✅ `torch.linalg.svd` (use the thin SVD: `full_matrices=False`).
- ❌ Don't form the covariance / Gram matrix $X_c^{\top}X_c$ in `pca_fit` (see *Why it works* in the solution).
- ❌ No `torch.pca_lowrank`, `torch.svd_lowrank` or scikit-learn.
- ❌ Don't whiten (rescale) in `pca_transform`.

---

<details>
  <summary>💡 Hint</summary>

  - Why must you subtract the mean first? Where would the "direction of maximum variance" point if
    the data sat far from the origin?
  - Substitute $X_c = U S V^{\top}$ into $X_c^{\top} X_c$. What are its eigenvectors and eigenvalues?
  - Of $U$ (size $N \times r$) and $V$ (size $D \times r$), which one lives in feature space?
    `torch.linalg.svd` returns $V^{\top}$, already sorted by decreasing singular value.
  - $X_c V = U S$. What does that say about the covariance of the projected data?

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Anisotropic Gaussian: known standard deviations along a random rotation, plus an offset mean.
torch.manual_seed(0)
N, D = 500, 5
true_std = torch.tensor([4.0, 2.0, 1.0, 0.5, 0.1])
Q, _ = torch.linalg.qr(torch.randn(D, D))                      # random orthonormal basis
X = torch.randn(N, D) * true_std @ Q.T + torch.tensor([1.0, -2.0, 0.5, 3.0, 0.0])
print("X", tuple(X.shape), "| per-direction variances ~", [round(v, 3) for v in (true_std ** 2).tolist()])

In [ ]:
def pca_fit(X, k):
    """
    Args:
        X: (N, D) data, one sample per row.
        k: number of components, 1 <= k <= min(N, D).
    Returns:
        components:         (k, D) principal directions as orthonormal rows, by decreasing variance.
        explained_variance: (k,) variance along each component (unbiased, divides by N - 1).
        mean:               (D,) per-feature mean.
    """
    # TODO: center, thin SVD, take the top-k right singular vectors and s^2 / (N - 1)
    raise NotImplementedError


def pca_transform(X, components, mean):
    """(N, D) -> (N, k) coordinates in the principal basis."""
    # TODO
    raise NotImplementedError


def pca_inverse_transform(Z, components, mean):
    """(N, k) -> (N, D) reconstruction in the original space."""
    # TODO
    raise NotImplementedError

In [ ]:
# Test 1: explained variance = top-k eigenvalues of the sample covariance (oracle: eigh)
k = 3
components, explained_variance, mean = pca_fit(X, k)
assert components.shape == (k, D) and explained_variance.shape == (k,) and mean.shape == (D,)

evals, evecs = torch.linalg.eigh(torch.cov(X.T))      # ascending order; torch.cov divides by N - 1
evals, evecs = evals.flip(0), evecs.flip(1)           # largest first
print("explained_variance:", explained_variance)
print("top-k eigenvalues :", evals[:k])
assert torch.allclose(explained_variance, evals[:k], rtol=1e-3)
assert torch.allclose(components @ components.T, torch.eye(k), atol=1e-5), "rows must be orthonormal"
cos = (components * evecs[:, :k].T).sum(1).abs()     # equal up to sign
assert torch.allclose(cos, torch.ones(k), atol=1e-3), f"components != eigenvectors: |cos| = {cos}"
print("Test 1 passed")

In [ ]:
# Test 2: projection, reconstruction error, exact reconstruction at k = D
k = 3
components, explained_variance, mean = pca_fit(X, k)
evals = torch.linalg.eigvalsh(torch.cov(X.T)).flip(0)   # covariance eigenvalues, largest first
Z = pca_transform(X, components, mean)
assert Z.shape == (N, k)
assert torch.allclose(torch.cov(Z.T), torch.diag(explained_variance), atol=1e-3 * explained_variance[0].item()), \
    "projected coordinates should be uncorrelated with variance explained_variance"
assert pca_transform(mean[None], components, mean).abs().max() < 1e-5, "the mean must map to 0"

X_hat = pca_inverse_transform(Z, components, mean)
err = ((X - X_hat) ** 2).sum() / (N - 1)
print(f"reconstruction error {err:.5f} = discarded variance {evals[k:].sum():.5f}")
assert torch.isclose(err, evals[k:].sum(), rtol=1e-3)

C_all, _, m_all = pca_fit(X, D)
X_back = pca_inverse_transform(pca_transform(X, C_all, m_all), C_all, m_all)
assert torch.allclose(X_back, X, atol=1e-4), "k = D must reconstruct X exactly"
print("Test 2 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("pca", pca_fit=pca_fit, pca_transform=pca_transform, pca_inverse_transform=pca_inverse_transform)